# XGBoost, LightGBM, CatBoost compared

### Three libraries, one algorithm, and a fair way to tell them apart

**Made by Elyes Lounissi** ·
[LinkedIn](https://www.linkedin.com/in/elyes-lounissi/) ·
[pilot.tun@gmail.com](mailto:pilot.tun@gmail.com) ·
[all notebooks](../../CURRICULUM.md)

---

| | |
|---|---|
| **What you will learn** | Why histogram binning made boosting fast, how level-wise, leaf-wise and oblivious tree growth differ, what native categorical handling buys you over one-hot encoding, and how much of the gap between these libraries is the library rather than the settings you gave it |
| **You should already know** | [Gradient boosting from first principles](../05-gradient-boosting/) |
| **Datasets** | UCI Dry Bean (13,611 x 16), UCI Bike Sharing (17,379 x 12) |
| **Runtime** | Two to four minutes on a laptop CPU, depending on which libraries you have installed |
| **Next** | [04-07 Stacking and voting](../07-stacking-and-voting/) |

---

## 1. What actually differs

The [previous notebook](../05-gradient-boosting/) built gradient boosting from
scratch in about fifteen lines and then used scikit-learn's original
`GradientBoostingRegressor`. That implementation sorts every feature at every
node to find the best split. It is correct, it is slow, and it is the reason
boosting had a reputation for being impractical on anything large.

Then XGBoost arrived in 2014, LightGBM in 2016, CatBoost in 2017, and
scikit-learn added `HistGradientBoosting` in 2019 after watching what the others
did. All four implement the same algorithm. The mathematics from the last
notebook is unchanged: fit a tree to the negative gradient, take a fraction of
the step, repeat.

What changed is engineering, and it comes down to four decisions:

1. **How continuous features are searched.** Exact sorting, or binned into a
   fixed number of buckets first.
2. **How the tree is grown.** Level by level, best-leaf first, or with one
   shared condition per level.
3. **How categorical columns are handled.** You encode them yourself, or the
   library splits on category sets directly.
4. **What the defaults are.** This turns out to matter more than the first
   three, and section 5 measures it.

Everything else — regularised leaf weights, column subsampling, early stopping,
missing-value routing — every one of these libraries has, and they behave much
the same way.

In [ ]:
import sys
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "toolkit").is_dir())
sys.path.insert(0, str(ROOT))

from toolkit import datasets, style

style.use()
FIG = pathlib.Path("figures")
SEED = 0

### Which of them do you actually have

Only `HistGradientBoosting` is guaranteed, because it ships inside scikit-learn.
The other three are separate installs, so this notebook checks for each one and
runs the comparison over whichever it finds. Every chart below renders either
way — a missing library is simply a missing series.

In [ ]:
import time
import warnings

import sklearn

# LightGBM and CatBoost are chatty about parameter aliases and about pandas
# category dtypes. None of it changes a result, and it drowns out the prints.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


def _optional(module_name):
    """Import a library if it is installed, otherwise return None."""
    try:
        module = __import__(module_name)
        return module, getattr(module, "__version__", "unknown")
    except ImportError:
        return None, None


xgb, XGB_VERSION = _optional("xgboost")
lgb, LGB_VERSION = _optional("lightgbm")
cb, CB_VERSION = _optional("catboost")

LIBRARIES = {
    "HistGB": {
        "found": True,
        "version": sklearn.__version__,
        "install": "ships with scikit-learn",
        "growth": "leaf-wise",
        "native categorical": "yes",
    },
    "XGBoost": {
        "found": xgb is not None,
        "version": XGB_VERSION or "-",
        "install": "pip install xgboost",
        "growth": "level-wise by default",
        "native categorical": "yes, opt in",
    },
    "LightGBM": {
        "found": lgb is not None,
        "version": LGB_VERSION or "-",
        "install": "pip install lightgbm",
        "growth": "leaf-wise",
        "native categorical": "yes, automatic",
    },
    "CatBoost": {
        "found": cb is not None,
        "version": CB_VERSION or "-",
        "install": "pip install catboost",
        "growth": "oblivious",
        "native categorical": "yes, ordered statistics",
    },
}

CONTENDERS = [name for name, spec in LIBRARIES.items() if spec["found"]]
MISSING = [name for name, spec in LIBRARIES.items() if not spec["found"]]

table = pd.DataFrame([
    {"library": name,
     "installed": "yes" if spec["found"] else "no",
     "version": spec["version"],
     "tree growth": spec["growth"],
     "categorical": spec["native categorical"],
     "get it with": spec["install"]}
    for name, spec in LIBRARIES.items()
])
print(table.to_string(index=False))
print()
print(f"comparing: {', '.join(CONTENDERS)}")
if MISSING:
    print(f"absent   : {', '.join(MISSING)}")
    print("Every figure still renders. To reproduce the full comparison, run:")
    print("    pip install " + " ".join(n.lower() for n in MISSING if n != "HistGB"))
else:
    print("all four present, so every chart below has four series")

## 2. Histogram binning

This is the change that made boosting fast, and it is worth understanding
properly because everything else is a smaller effect.

### The exact method

To split a node on a continuous feature, the honest thing to do is try every
threshold that separates two adjacent values. With $n$ rows in the node that is
up to $n - 1$ candidates, and to evaluate them cheaply you first sort the column
so that a running sum of gradients gives you both sides of every split in one
pass. Sorting costs $O(n \log n)$, and you pay it for every feature at every
node.

### The binned method

Before training starts, each continuous feature is chopped into a small fixed
number of buckets — 255 by default almost everywhere — using quantiles of the
column, and every value is replaced by its bucket number. That is a one-off
cost paid once for the whole dataset.

Now a node's split search is: sweep the rows once adding each gradient into its
bucket, then scan the buckets in order. The sort is gone. The scan is over
`max_bin` positions instead of $n$ of them, and `max_bin` does not grow with
your data.

There is a second trick on top. A node's two children partition its rows, so
the histograms of the two children sum to the histogram of the parent. Build
the histogram for the smaller child, subtract it from the parent, and you get
the larger child for free. Half the work at every level disappears.

Let me measure all three costs.

In [ ]:
def best_split_exact(x, grad):
    """Sort the column, then evaluate every boundary with a running sum."""
    order = np.argsort(x, kind="quicksort")
    ordered = grad[order]
    left_sum = np.cumsum(ordered)[:-1]
    left_n = np.arange(1, len(ordered), dtype=np.float64)
    right_sum = ordered.sum() - left_sum
    right_n = len(ordered) - left_n
    # Variance-reduction gain, dropping the constant total term.
    gain = left_sum ** 2 / left_n + right_sum ** 2 / right_n
    return float(gain.max())


def build_histogram(codes, grad, n_bins):
    """One pass over the rows. This is the part the subtraction trick avoids."""
    return (np.bincount(codes, weights=grad, minlength=n_bins),
            np.bincount(codes, minlength=n_bins).astype(np.float64))


def best_split_from_histogram(hist_sum, hist_count):
    """Scan the buckets in order. Costs the same whatever n was."""
    left_sum = np.cumsum(hist_sum)[:-1]
    left_n = np.cumsum(hist_count)[:-1]
    right_sum = hist_sum.sum() - left_sum
    right_n = hist_count.sum() - left_n
    with np.errstate(divide="ignore", invalid="ignore"):
        gain = np.where(left_n * right_n > 0,
                        left_sum ** 2 / np.maximum(left_n, 1)
                        + right_sum ** 2 / np.maximum(right_n, 1), -np.inf)
    return float(gain.max())


def timed(fn, repeat=3):
    """Best of `repeat` runs, which is the fair way to time on a busy laptop."""
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - start)
    return best


MAX_BIN = 255
rng = np.random.default_rng(SEED)
sizes = [10_000, 30_000, 100_000, 300_000, 1_000_000]
cost = {"exact sort and scan": [], "build histogram and scan": [], "scan only": []}

for n in sizes:
    x = rng.standard_normal(n)
    grad = rng.standard_normal(n)

    # Binning is done once, before the first tree, so it is not in the per-node cost.
    edges = np.unique(np.quantile(x, np.linspace(0, 1, MAX_BIN + 1)[1:-1]))
    codes = np.searchsorted(edges, x).astype(np.int32)
    n_bins = len(edges) + 1
    hist_sum, hist_count = build_histogram(codes, grad, n_bins)

    cost["exact sort and scan"].append(timed(lambda: best_split_exact(x, grad)))
    cost["build histogram and scan"].append(
        timed(lambda: best_split_from_histogram(*build_histogram(codes, grad, n_bins))))
    cost["scan only"].append(
        timed(lambda: best_split_from_histogram(hist_sum, hist_count)))

for label, times in cost.items():
    row = "  ".join(f"{t * 1000:8.3f}" for t in times)
    print(f"{label:<26} {row}   ms, at n = {', '.join(f'{s:,}' for s in sizes)}")

speedup = cost["exact sort and scan"][-1] / cost["build histogram and scan"][-1]
scan_only = cost["exact sort and scan"][-1] / cost["scan only"][-1]
print(f"\nat one million rows the histogram path is {speedup:.1f}x faster than sorting,")
print(f"and {scan_only:.0f}x faster when the histogram came free by subtraction")

In [ ]:
X_bean, y_bean = datasets.dry_bean()
unique_counts = X_bean.nunique().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.4))

positions = np.arange(len(unique_counts))
axes[0].hlines(positions, np.minimum(unique_counts.values, MAX_BIN), unique_counts.values,
               color=style.RULE, lw=1.4, zorder=1)
axes[0].scatter(unique_counts.values, positions, color=style.NEUTRAL,
                marker=style.MARKERS[0], s=34, zorder=2, label="exact: every distinct value")
axes[0].scatter(np.minimum(unique_counts.values, MAX_BIN), positions, color=style.HIGHLIGHT,
                marker=style.MARKERS[1], s=34, zorder=3, label=f"binned: at most {MAX_BIN}")
axes[0].set_yticks(positions, list(unique_counts.index), fontsize=8)
axes[0].set_xscale("log")
axes[0].set_xlabel("candidate split points in one node (log scale)")
axes[0].grid(axis="y", visible=False)
axes[0].grid(axis="x", visible=True)
axes[0].legend(loc="lower right")
style.title(axes[0], "Binning shrinks the search by two orders of magnitude",
            "UCI Dry Bean, 16 features, at the root node")

for i, (label, times) in enumerate(cost.items()):
    axes[1].plot(sizes, [t * 1000 for t in times], marker=style.MARKERS[i],
                 color=style.HIGHLIGHT if i == 0 else style.PALETTE[i + 1], label=label)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("rows in the node (log scale)")
axes[1].set_ylabel("time to find the best split, ms (log scale)")
axes[1].legend()
style.title(axes[1], "The sort is what costs, and binning deletes it",
            "one feature, NumPy, best of three runs")

style.save(fig, FIG / "fig-01-histogram-binning.png")

The left panel is the reason this works at all. Most of these bean measurements
are continuous, so nearly every row carries a distinct value and the exact
method has thousands of thresholds to consider per feature per node. Binning
caps that at 255 regardless.

The right panel separates the two costs that binning replaces. Building the
histogram is still one pass over the rows, so it grows with $n$ — but it is a
single cheap pass with no comparisons, and the flat line underneath it is what
a node costs when its histogram arrived by subtracting a sibling.

The obvious question is what accuracy this throws away. Fewer bins means
coarser thresholds, so the answer should be "a little, and less as bins go up".

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# XGBoost wants integer class labels, so encode once and use the codes everywhere.
bean_labels = LabelEncoder().fit(y_bean)
y_bean_coded = pd.Series(bean_labels.transform(y_bean), name="Class")

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_bean, y_bean_coded, test_size=0.25, random_state=SEED, stratify=y_bean_coded)


def classifier(library, *, n_estimators=200, learning_rate=0.1, num_leaves=31,
               max_bin=255, seed=SEED):
    """One configuration, translated into each library's own vocabulary.

    The point of this function is that the four libraries use four different
    names for the same four dials, which is most of why casual comparisons
    between them are unfair.
    """
    if library == "HistGB":
        return HistGradientBoostingClassifier(
            max_iter=n_estimators, learning_rate=learning_rate,
            max_leaf_nodes=num_leaves, max_bins=min(max_bin, 255),
            early_stopping=False, random_state=seed)
    if library == "LightGBM":
        return lgb.LGBMClassifier(
            n_estimators=n_estimators, learning_rate=learning_rate,
            num_leaves=num_leaves, max_bin=max_bin,
            random_state=seed, n_jobs=-1, verbose=-1)
    if library == "XGBoost":
        # max_depth=0 plus lossguide is how XGBoost is asked to grow leaf-wise.
        return xgb.XGBClassifier(
            n_estimators=n_estimators, learning_rate=learning_rate,
            max_leaves=num_leaves, max_depth=0, grow_policy="lossguide",
            max_bin=max_bin, tree_method="hist",
            random_state=seed, n_jobs=-1, verbosity=0)
    if library == "CatBoost":
        return cb.CatBoostClassifier(
            iterations=n_estimators, learning_rate=learning_rate,
            max_leaves=num_leaves, grow_policy="Lossguide",
            border_count=min(max_bin, 254), random_seed=seed,
            verbose=0, thread_count=-1, allow_writing_files=False)
    raise ValueError(library)


print(f"{'library':<10} {'max_bin':>8} {'accuracy':>10} {'fit seconds':>13}")
for library in CONTENDERS:
    for max_bin in [15, 63, 255]:
        model = classifier(library, n_estimators=100, max_bin=max_bin)
        start = time.perf_counter()
        model.fit(Xb_train, yb_train)
        elapsed = time.perf_counter() - start
        accuracy = (model.predict(Xb_test) == yb_test.values).mean()
        print(f"{library:<10} {max_bin:>8} {accuracy:>10.4f} {elapsed:>13.2f}")

Coarse bins cost accuracy and buy time, and the curve flattens quickly. That is
why nobody tunes `max_bin` in practice: the default is already past the knee.
The one time it is worth turning down is when the dataset is too big to be
comfortable, because bins also decide how much memory the binned matrix needs.

## 3. Level-wise against leaf-wise

Given a budget of splits, where do you spend them?

**Level-wise**, which is XGBoost's default: split every node on the current
level before starting the next one. The tree stays balanced and `max_depth`
controls it exactly.

**Leaf-wise**, which is LightGBM's only mode and also what scikit-learn's
`HistGradientBoosting` does: keep a queue of leaves ordered by the gain they
would produce if split, and always split the best one. The tree comes out
lopsided, sometimes very deep down one branch and shallow everywhere else.

**Oblivious**, which is CatBoost's default: pick one split condition per level
and apply it at every node on that level. The tree is perfectly balanced and
every leaf is reached by the same sequence of yes/no answers, which means
prediction is a bit-shift into an array rather than a walk down pointers. That
is why CatBoost predicts so fast. It also acts as heavy regularisation, because
the tree has far less freedom than the other two.

For the same number of splits, leaf-wise reduces training loss the most,
because it spends every split where the gain is largest. That is also exactly
why it is the easiest of the three to overfit with.

In [ ]:
def draw_tree(ax, children, headline, sub, level_labels=None):
    """Draw a small binary tree. Leaves are placed left to right, parents at the
    midpoint of their children, which keeps any shape readable."""
    pos, counter = {}, [0]

    def place(node, depth):
        kids = children.get(node)
        if not kids:
            pos[node] = (counter[0], -depth)
            counter[0] += 1
            return pos[node][0]
        left = place(kids[0], depth + 1)
        right = place(kids[1], depth + 1)
        pos[node] = ((left + right) / 2, -depth)
        return pos[node][0]

    place(0, 0)
    internal = set(children)

    for parent, kids in children.items():
        for kid in kids:
            ax.plot([pos[parent][0], pos[kid][0]], [pos[parent][1], pos[kid][1]],
                    color=style.RULE, lw=1.3, zorder=1, solid_capstyle="round")
    for node, (x, y) in pos.items():
        leaf = node not in internal
        ax.scatter([x], [y], s=95 if leaf else 80, zorder=2,
                   marker="s" if leaf else "o",
                   color=style.HIGHLIGHT if leaf else style.NEUTRAL,
                   edgecolor="white", linewidth=0.9)

    depth = int(max(-y for _, y in pos.values()))
    n_leaves = len(pos) - len(internal)
    ax.annotate(f"{len(internal)} splits, {n_leaves} leaves, depth {depth}",
                xy=(0.0, 0.0), xycoords="axes fraction", xytext=(0, -6),
                textcoords="offset points", ha="left", va="top",
                fontsize=9, color=style.MUTED)

    if level_labels:
        right = max(x for x, _ in pos.values())
        for level, text in enumerate(level_labels):
            ax.text(right + 0.4, -level, text, fontsize=8.5, color=style.PALETTE[0],
                    va="center", ha="left", family="monospace")

    ax.set_xlim(-0.7, max(x for x, _ in pos.values()) + (3.4 if level_labels else 0.7))
    ax.set_ylim(-depth - 0.6, 0.6)
    ax.axis("off")
    style.title(ax, headline, sub)


# Same budget of seven splits in all three, spent differently.
level_wise = {0: (1, 2), 1: (3, 4), 2: (5, 6), 3: (7, 8), 4: (9, 10), 5: (11, 12), 6: (13, 14)}
leaf_wise = {0: (1, 2), 1: (3, 4), 3: (5, 6), 5: (7, 8), 7: (9, 10), 9: (11, 12), 11: (13, 14)}

fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.3))
draw_tree(axes[0], level_wise, "Level-wise", "XGBoost by default: finish a level, then descend")
draw_tree(axes[1], leaf_wise, "Leaf-wise", "LightGBM and HistGB: always split the best leaf")
draw_tree(axes[2], level_wise, "Oblivious", "CatBoost by default: one condition for a whole level",
          level_labels=["area < 41,900", "roundness < 0.79", "aspect < 1.68"])
fig.suptitle("Seven splits, three ways to spend them",
             x=0.5, y=1.04, fontsize=12.5, color=style.INK)
style.save(fig, FIG / "fig-02-growth-strategies.png")

### What leaf-wise costs you on small data

A leaf-wise tree with 31 leaves can be 30 levels deep down one branch, and the
leaf at the bottom of that branch may be holding a handful of rows. On a large
dataset that is fine, because a deep branch still has plenty of rows in it. On
a small dataset it is memorisation.

This is why LightGBM's documentation tells you to lower `num_leaves` on small
data, and it is the single most common way to get a bad result out of it. Below
I train on a deliberately small slice and let `num_leaves` grow.

In [ ]:
small_train, small_test, small_y_train, small_y_test = train_test_split(
    X_bean, y_bean_coded, train_size=900, test_size=3000,
    random_state=SEED, stratify=y_bean_coded)

leaf_counts = [4, 8, 16, 31, 63, 127, 255]
print(f"{'library':<10} {'num_leaves':>11} {'train':>8} {'held out':>10} {'gap':>8}")
for library in CONTENDERS:
    for leaves in leaf_counts:
        model = classifier(library, n_estimators=150, num_leaves=leaves)
        model.fit(small_train, small_y_train)
        train_acc = (model.predict(small_train) == small_y_train.values).mean()
        test_acc = (model.predict(small_test) == small_y_test.values).mean()
        print(f"{library:<10} {leaves:>11} {train_acc:>8.4f} {test_acc:>10.4f} "
              f"{train_acc - test_acc:>8.4f}")
    print()

print("On 900 rows the training accuracy walks up to a perfect score while the")
print("held-out accuracy does not follow. The gap is the overfitting.")

## 4. Categorical features

A tree splits on "is this value below that threshold". A category has no
below. So something has to give, and there are three things people do.

**Ordinal codes.** Leave the integer codes in place and let the tree split on
them as if they were numbers. Cheap, and it invents an ordering that is not
there: it lets the tree say "weekday < 3", which groups Sunday, Monday and
Tuesday for no reason other than their labels.

**One-hot.** One binary column per level. Honest, and it makes the tree's job
harder: to isolate a group of five categories out of twenty, the tree needs
five separate splits, each one on a column that is mostly zero and therefore
has little gain on its own. Columns also multiply, and with them memory and
fitting time.

**Native.** The library knows the column is categorical and splits it into two
*sets* of categories in one go. LightGBM does this by sorting the categories by
their accumulated gradient and then scanning that order as if it were numeric,
which finds a good set partition in one pass. CatBoost goes further, and
section 4.2 covers what it does.

Bike Sharing is the right dataset for this, because `season`, `mnth`,
`weekday` and `weathersit` are genuine labels that arrive as integers and are
therefore very easy to feed a model as if they were quantities.

In [ ]:
X_bike, y_bike = datasets.bike_sharing()
CATEGORICAL = ["season", "mnth", "weekday", "weathersit"]

print("Bike Sharing columns and their distinct values")
print(X_bike.nunique().to_string())
print(f"\ntreating as categorical: {', '.join(CATEGORICAL)}")
print(f"one-hot would turn those {len(CATEGORICAL)} columns into "
      f"{int(X_bike[CATEGORICAL].nunique().sum())}")


def encode(frame, mode, library):
    """Shape the frame the way this library expects for this encoding."""
    if mode == "ordinal":
        return frame.astype(float)
    if mode == "one-hot":
        return pd.get_dummies(frame, columns=CATEGORICAL, dtype=float)
    out = frame.copy()
    for column in CATEGORICAL:
        # CatBoost wants the raw labels; the others want a pandas category dtype.
        out[column] = (out[column].astype(str) if library == "CatBoost"
                       else out[column].astype("category"))
    return out


def regressor(library, mode, *, n_estimators=300, learning_rate=0.1, num_leaves=31):
    native = mode == "native"
    if library == "HistGB":
        return HistGradientBoostingRegressor(
            max_iter=n_estimators, learning_rate=learning_rate,
            max_leaf_nodes=num_leaves, categorical_features=CATEGORICAL if native else None,
            early_stopping=False, random_state=SEED)
    if library == "LightGBM":
        # LightGBM picks up category dtype columns on its own, so nothing to declare.
        return lgb.LGBMRegressor(
            n_estimators=n_estimators, learning_rate=learning_rate,
            num_leaves=num_leaves, random_state=SEED, n_jobs=-1, verbose=-1)
    if library == "XGBoost":
        return xgb.XGBRegressor(
            n_estimators=n_estimators, learning_rate=learning_rate,
            max_leaves=num_leaves, max_depth=0, grow_policy="lossguide",
            tree_method="hist", enable_categorical=native,
            random_state=SEED, n_jobs=-1, verbosity=0)
    if library == "CatBoost":
        return cb.CatBoostRegressor(
            iterations=n_estimators, learning_rate=learning_rate,
            max_leaves=num_leaves, grow_policy="Lossguide",
            cat_features=CATEGORICAL if native else None,
            random_seed=SEED, verbose=0, thread_count=-1, allow_writing_files=False)
    raise ValueError(library)


MODES = ["ordinal", "one-hot", "native"]
cat_results = []

for library in CONTENDERS:
    for mode in MODES:
        frame = encode(X_bike, mode, library)
        train_X, test_X, train_y, test_y = train_test_split(
            frame, y_bike, test_size=0.25, random_state=SEED)
        model = regressor(library, mode)
        start = time.perf_counter()
        model.fit(train_X, train_y)
        fit_seconds = time.perf_counter() - start
        rmse = float(np.sqrt(np.mean((test_y.values - model.predict(test_X)) ** 2)))
        cat_results.append({"library": library, "encoding": mode, "columns": frame.shape[1],
                            "RMSE": rmse, "fit seconds": fit_seconds})

cat_frame = pd.DataFrame(cat_results)
print(cat_frame.round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.3))
width = 0.26
positions = np.arange(len(CONTENDERS))

for i, mode in enumerate(MODES):
    subset = cat_frame[cat_frame["encoding"] == mode].set_index("library")
    offset = (i - 1) * width
    colour = style.HIGHLIGHT if mode == "native" else style.PALETTE[i]
    axes[0].bar(positions + offset, [subset.loc[n, "RMSE"] for n in CONTENDERS],
                width, color=colour, label=mode)
    axes[1].bar(positions + offset, [subset.loc[n, "fit seconds"] for n in CONTENDERS],
                width, color=colour, label=mode)

axes[0].set_xticks(positions, CONTENDERS)
axes[0].set_ylabel("held-out RMSE, hires per hour")
axes[0].legend(title="categorical columns as")
style.title(axes[0], "Telling the library which columns are labels helps",
            "UCI Bike Sharing, 300 trees, 31 leaves, same settings everywhere")

axes[1].set_xticks(positions, CONTENDERS)
axes[1].set_ylabel("training time, seconds")
axes[1].legend(title="categorical columns as")
style.title(axes[1], "And one-hot is the one that costs time",
            "wider matrix, more columns to scan at every node")

style.save(fig, FIG / "fig-03-categorical.png")

Two things to take from that, and the second is the more useful one.

Native handling wins on error, because a single split on a set of categories is
something the tree can find in one step and would need several splits to
approximate through one-hot columns.

But the timing gap is small here, and I want to be straight about why: the
widest column in Bike Sharing has twelve levels, so one-hot adds a couple of
dozen columns and nothing more. One-hot becomes expensive when cardinality is
in the hundreds or thousands — a postcode, a product id, a user id. So let me
build exactly that case out of columns this dataset already has, by crossing
month with hour of day.

In [ ]:
X_wide = X_bike.copy()
X_wide["mnth_hr"] = X_wide["mnth"] * 100 + X_wide["hr"]   # a label, not a quantity
WIDE_CATEGORICAL = CATEGORICAL + ["mnth_hr"]
print(f"the crossed column has {X_wide['mnth_hr'].nunique()} levels")

wide_results = []
for library in CONTENDERS:
    for mode in ["one-hot", "native"]:
        if mode == "one-hot":
            frame = pd.get_dummies(X_wide, columns=WIDE_CATEGORICAL, dtype=float)
        else:
            frame = X_wide.copy()
            for column in WIDE_CATEGORICAL:
                frame[column] = (frame[column].astype(str) if library == "CatBoost"
                                 else frame[column].astype("category"))
        train_X, test_X, train_y, test_y = train_test_split(
            frame, y_bike, test_size=0.25, random_state=SEED)

        # Reuse the regressor factory but declare the crossed column too.
        saved, globals()["CATEGORICAL"] = CATEGORICAL, WIDE_CATEGORICAL
        model = regressor(library, mode, n_estimators=200)
        globals()["CATEGORICAL"] = saved

        start = time.perf_counter()
        model.fit(train_X, train_y)
        fit_seconds = time.perf_counter() - start
        rmse = float(np.sqrt(np.mean((test_y.values - model.predict(test_X)) ** 2)))
        wide_results.append({"library": library, "encoding": mode,
                             "columns": frame.shape[1], "RMSE": rmse,
                             "fit seconds": fit_seconds})

wide_frame = pd.DataFrame(wide_results)
print(wide_frame.round(3).to_string(index=False))
for library in CONTENDERS:
    rows = wide_frame[wide_frame["library"] == library].set_index("encoding")
    ratio = rows.loc["one-hot", "fit seconds"] / rows.loc["native", "fit seconds"]
    print(f"{library:<10} one-hot took {ratio:.1f}x the time of native handling")

### 4.2 CatBoost and the leakage in target encoding

There is a fourth encoding I skipped, and it is the one that gets people into
trouble.

**Target encoding** replaces a category with the mean target for that category.
It is compact, it carries real signal, and it leaks. The row you are encoding
contributed to the mean you are encoding it with, so the feature contains a
piece of that row's own answer. On a category with two rows in it, the encoded
value is roughly half the answer. Training accuracy goes up, held-out accuracy
goes down, and the model looks fine until it ships.

The usual patch is to compute the means out of fold, or to hold out the current
row when averaging. CatBoost's answer is called **ordered target statistics**
and it is tidier. It draws a random permutation of the rows and treats it as if
it were time: when it encodes row $i$, it averages the target over only the
rows that came before $i$ in that permutation.

$$\hat{x}_i = \frac{\sum_{j < i} [x_j = x_i] \, y_j + a \, p}{\sum_{j < i} [x_j = x_i] + a}$$

The $a\,p$ term is a prior with weight $a$, which stops a category seen once
from taking that single target as its value. Row $i$ never contributes to its
own encoding, so the leak is closed by construction. Several permutations are
kept and rotated so that early rows in any one ordering are not stuck with
noisy estimates.

The same idea applied to the gradients rather than the features is what
CatBoost calls **ordered boosting**, and it is where its reputation on small
datasets comes from. It costs time, which is part of why CatBoost is usually
the slowest of the three to fit.

Here is the leak, measured. No CatBoost needed — the point is about the
encoding, and I can build both versions by hand.

In [ ]:
leak_frame = X_bike.copy()
leak_frame["mnth_hr"] = leak_frame["mnth"] * 100 + leak_frame["hr"]

leak_train, leak_test, leak_y_train, leak_y_test = train_test_split(
    leak_frame, y_bike, train_size=1500, test_size=4000, random_state=SEED)

prior = leak_y_train.mean()

# The naive version: one mean per category, computed on all of the training rows.
naive_map = leak_y_train.groupby(leak_train["mnth_hr"]).mean()

# The ordered version: for each row, average only over the rows before it.
order = np.random.default_rng(SEED).permutation(len(leak_train))
running_sum, running_count = {}, {}
ordered_encoded = np.empty(len(leak_train))
SMOOTHING = 10.0
for position in order:
    category = leak_train["mnth_hr"].iloc[position]
    total, count = running_sum.get(category, 0.0), running_count.get(category, 0)
    ordered_encoded[position] = (total + SMOOTHING * prior) / (count + SMOOTHING)
    running_sum[category] = total + leak_y_train.iloc[position]
    running_count[category] = count + 1

test_encoded = leak_test["mnth_hr"].map(naive_map).fillna(prior).values

for label, encoded_train in [("naive target encoding", leak_train["mnth_hr"].map(naive_map).values),
                             ("ordered target statistics", ordered_encoded)]:
    train_X = leak_train.drop(columns="mnth_hr").astype(float).assign(encoded=encoded_train)
    test_X = leak_test.drop(columns="mnth_hr").astype(float).assign(encoded=test_encoded)
    model = HistGradientBoostingRegressor(max_iter=200, max_leaf_nodes=31,
                                          early_stopping=False, random_state=SEED)
    model.fit(train_X, leak_y_train)
    train_rmse = float(np.sqrt(np.mean((leak_y_train.values - model.predict(train_X)) ** 2)))
    test_rmse = float(np.sqrt(np.mean((leak_y_test.values - model.predict(test_X)) ** 2)))
    print(f"{label:<28} train RMSE {train_rmse:7.2f}   held out {test_rmse:7.2f}   "
          f"gap {test_rmse - train_rmse:6.2f}")

print("\nThe naive version looks better in training and is not better where it counts.")

## 5. The benchmark, and how much is the library rather than the tuning

Now the comparison everyone wants, done in the way that makes it mean
something.

Most benchmarks you find online compare the libraries at their own defaults.
That measures the defaults, not the libraries, and the defaults differ wildly:
LightGBM builds 100 trees of 31 leaves, XGBoost builds 100 trees of depth 6,
CatBoost builds a thousand trees. Comparing those three is comparing three
different amounts of model.

So I run it twice. Once at each library's own defaults, which is the number you
get if you type three lines and stop. Once with the same tree count, learning
rate, leaf budget and bin count forced on all of them, which is the number that
tells you about the implementation.

In [ ]:
def default_classifier(library):
    if library == "HistGB":
        return HistGradientBoostingClassifier(random_state=SEED)
    if library == "LightGBM":
        return lgb.LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1)
    if library == "XGBoost":
        return xgb.XGBClassifier(random_state=SEED, n_jobs=-1, verbosity=0)
    if library == "CatBoost":
        # True default is 1000 iterations. Capped here so this notebook stays
        # under five minutes, and it is worth knowing that the default is slow.
        return cb.CatBoostClassifier(iterations=300, random_seed=SEED, verbose=0,
                                     thread_count=-1, allow_writing_files=False)
    raise ValueError(library)


def benchmark(model, train_X, train_y, test_X, test_y):
    start = time.perf_counter()
    model.fit(train_X, train_y)
    fit_seconds = time.perf_counter() - start

    start = time.perf_counter()
    predictions = model.predict(test_X)
    predict_seconds = time.perf_counter() - start

    return {"accuracy": float((predictions == test_y.values).mean()),
            "fit seconds": fit_seconds,
            "predict microseconds per row": predict_seconds / len(test_X) * 1e6}


rows = []
for library in CONTENDERS:
    for setting, model in [("its own defaults", default_classifier(library)),
                           ("matched settings", classifier(library, n_estimators=200,
                                                           learning_rate=0.1,
                                                           num_leaves=31, max_bin=255))]:
        result = benchmark(model, Xb_train, yb_train, Xb_test, yb_test)
        rows.append({"library": library, "setting": setting, **result})

bench = pd.DataFrame(rows)
print(bench.round(4).to_string(index=False))

for setting in ["its own defaults", "matched settings"]:
    subset = bench[bench["setting"] == setting]
    spread = subset["accuracy"].max() - subset["accuracy"].min()
    print(f"\n{setting}: accuracy spread across libraries {spread:.4f}, "
          f"fastest fit {subset['fit seconds'].min():.2f}s, "
          f"slowest {subset['fit seconds'].max():.2f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.4))

matched = bench[bench["setting"] == "matched settings"].reset_index(drop=True)
defaults = bench[bench["setting"] == "its own defaults"].reset_index(drop=True)

for i, row in matched.iterrows():
    axes[0].scatter(row["fit seconds"], row["accuracy"], s=130,
                    color=style.PALETTE[i], marker=style.MARKERS[i], zorder=3,
                    label=row["library"])
    axes[0].annotate(row["library"], xy=(row["fit seconds"], row["accuracy"]),
                     xytext=(9, 5), textcoords="offset points",
                     fontsize=9.5, color=style.INK)
for i, row in defaults.iterrows():
    axes[0].scatter(row["fit seconds"], row["accuracy"], s=95, facecolor="white",
                    edgecolor=style.PALETTE[i], linewidth=1.6,
                    marker=style.MARKERS[i], zorder=2)
    partner = matched[matched["library"] == row["library"]].iloc[0]
    axes[0].plot([row["fit seconds"], partner["fit seconds"]],
                 [row["accuracy"], partner["accuracy"]],
                 color=style.RULE, lw=1.1, ls="--", zorder=1)

axes[0].set_xlabel("training time, seconds (filled = matched, hollow = defaults)")
axes[0].set_ylabel("held-out accuracy")
style.title(axes[0], "Same accuracy, different amounts of time to get there",
            "UCI Dry Bean, 10,208 training rows, 7 classes")

order = bench[bench["setting"] == "matched settings"].sort_values(
    "predict microseconds per row")
axes[1].barh(order["library"], order["predict microseconds per row"],
             color=[style.HIGHLIGHT] + [style.NEUTRAL] * (len(order) - 1), height=0.6)
for i, (_, row) in enumerate(order.iterrows()):
    axes[1].text(row["predict microseconds per row"] * 1.02, i,
                 f"{row['predict microseconds per row']:.1f}",
                 va="center", fontsize=9, color=style.MUTED)
axes[1].set_xlabel("microseconds per row")
axes[1].grid(axis="y", visible=False)
axes[1].grid(axis="x", visible=True)
style.title(axes[1], "Prediction cost is a separate question from fitting cost",
            "matched settings, 3,403 held-out rows")

style.save(fig, FIG / "fig-04-accuracy-vs-time.png")

### How they scale with rows

Fitting time on one dataset size tells you almost nothing, because the fixed
overheads differ. The shape of the curve is the useful part. Bike Sharing is
the larger of the two house datasets, and I resample it with replacement to get
past its natural size — that makes the accuracy meaningless, which is fine,
because this cell only measures time.

In [ ]:
row_counts = [2_500, 5_000, 10_000, 20_000, 40_000, 80_000, 160_000]
scaling = {library: [] for library in CONTENDERS}
sampler = np.random.default_rng(SEED)

X_scale = X_bike.astype(float).values
y_scale = y_bike.values

for n in row_counts:
    picks = sampler.integers(0, len(X_scale), n)
    Xn, yn = X_scale[picks], y_scale[picks]
    for library in CONTENDERS:
        model = regressor(library, "ordinal", n_estimators=100, num_leaves=31)
        start = time.perf_counter()
        model.fit(Xn, yn)
        scaling[library].append(time.perf_counter() - start)

fig, ax = plt.subplots(figsize=(8.6, 4.6))
for i, library in enumerate(CONTENDERS):
    ax.plot(row_counts, scaling[library], marker=style.MARKERS[i],
            color=style.PALETTE[i], label=library)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("training rows (log scale)")
ax.set_ylabel("time to fit 100 trees, seconds (log scale)")
ax.legend()
style.title(ax, "All of them are close to linear in rows",
            "UCI Bike Sharing resampled with replacement, 100 trees, 31 leaves")
style.save(fig, FIG / "fig-05-scaling.png")

print(f"{'library':<10} " + " ".join(f"{n:>9,}" for n in row_counts))
for library in CONTENDERS:
    print(f"{library:<10} " + " ".join(f"{t:>9.2f}" for t in scaling[library]))
print("\nseconds to fit 100 trees. Slopes on the log-log plot:")
for library in CONTENDERS:
    slope = np.polyfit(np.log(row_counts), np.log(scaling[library]), 1)[0]
    print(f"  {library:<10} {slope:.2f}   (1.00 would be exactly linear in rows)")

The binning is why these lines are as flat as they are. A slope near one means
doubling the rows doubles the time, which is the best you can hope for from
something that has to look at every row. The exact method in the
[previous notebook](../05-gradient-boosting/) has a $\log n$ factor on top of
that, plus a much larger constant.

### The part nobody puts in the benchmark

Here is the question I actually care about. The gap between these libraries at
matched settings is small. Is it larger or smaller than the gap between a bad
configuration and a good one *within* a single library?

I run the same seven configurations through every library and compare two
numbers: how much accuracy moves when you change the library and hold the
settings, against how much it moves when you change the settings and hold the
library.

In [ ]:
GRID = [
    {"n_estimators": 100, "learning_rate": 0.30, "num_leaves": 15, "max_bin": 255},
    {"n_estimators": 150, "learning_rate": 0.10, "num_leaves": 31, "max_bin": 255},
    {"n_estimators": 200, "learning_rate": 0.05, "num_leaves": 63, "max_bin": 255},
    {"n_estimators": 300, "learning_rate": 0.05, "num_leaves": 15, "max_bin": 127},
    {"n_estimators": 250, "learning_rate": 0.03, "num_leaves": 31, "max_bin": 255},
    {"n_estimators": 100, "learning_rate": 0.10, "num_leaves": 127, "max_bin": 63},
    {"n_estimators": 200, "learning_rate": 0.20, "num_leaves": 7, "max_bin": 255},
]

# A smaller slice, so seven configurations times however many libraries stays quick.
tune_train, tune_test, tune_y_train, tune_y_test = train_test_split(
    X_bean, y_bean_coded, train_size=5000, test_size=3500,
    random_state=SEED, stratify=y_bean_coded)

scores = np.zeros((len(GRID), len(CONTENDERS)))
for row, config in enumerate(GRID):
    for column, library in enumerate(CONTENDERS):
        model = classifier(library, **config)
        model.fit(tune_train, tune_y_train)
        scores[row, column] = (model.predict(tune_test) == tune_y_test.values).mean()

grid_frame = pd.DataFrame(scores, columns=CONTENDERS)
grid_frame.insert(0, "config", [
    f"{c['n_estimators']} trees, lr {c['learning_rate']}, {c['num_leaves']} leaves, "
    f"{c['max_bin']} bins" for c in GRID])
print(grid_frame.round(4).to_string(index=False))

tuning_spread = float(np.mean(scores.max(axis=0) - scores.min(axis=0)))
library_spread = float(np.mean(scores.max(axis=1) - scores.min(axis=1)))

print(f"\nchange the settings, hold the library : accuracy moves {tuning_spread:.4f} on average")
print(f"change the library, hold the settings : accuracy moves {library_spread:.4f} on average")
print(f"ratio: {tuning_spread / max(library_spread, 1e-9):.1f}x")
best_row, best_col = np.unravel_index(int(np.argmax(scores)), scores.shape)
print(f"\nbest overall: {CONTENDERS[best_col]} at {grid_frame['config'][best_row]} "
      f"({scores[best_row, best_col]:.4f})")
worst_row, worst_col = np.unravel_index(int(np.argmin(scores)), scores.shape)
print(f"worst overall: {CONTENDERS[worst_col]} at {grid_frame['config'][worst_row]} "
      f"({scores[worst_row, worst_col]:.4f})")

That ratio is the honest answer to "which one should I use", and it is not a
ranking of libraries.

Changing the configuration moves the accuracy by considerably more than
changing the library does. If you are choosing between XGBoost, LightGBM and
CatBoost on the basis of a leaderboard where they finish within a fraction of a
percent of each other, you are optimising the wrong thing. Pick the one whose
API you will not fight, and spend the time you saved on the learning rate and
the leaf budget.

Where they genuinely differ is on things that are not accuracy:

**Speed of fitting.** LightGBM is usually the fastest, and the gap grows with
rows. This matters when you are doing cross-validated search, because you pay
it dozens of times.

**Speed of prediction.** CatBoost's oblivious trees evaluate as an array index
rather than a tree walk, which is a real advantage if you are serving at
latency.

**Categorical columns.** If you have many of them and they are high
cardinality, CatBoost's ordered statistics is the least work for the best
result, and LightGBM's set splitting is close behind.

**Small datasets.** CatBoost's ordered boosting is designed exactly for the
case where the others overfit. LightGBM leaf-wise is the most dangerous here
unless you turn `num_leaves` down.

**Not installing anything.** `HistGradientBoosting` is already there, it is
within noise of the others, and it removes a dependency from your project. For
a lot of work that is the right trade.

## Cheat sheet

| Situation | Reach for |
|---|---|
| You want a strong tabular baseline in three lines and no new dependency | `HistGradientBoostingClassifier`, already installed |
| Millions of rows, or a search over hundreds of configurations | LightGBM, the fastest to fit |
| Many high-cardinality categorical columns | CatBoost, then LightGBM |
| A few thousand rows and you cannot afford to overfit | CatBoost, or LightGBM with `num_leaves` turned down hard |
| You are serving predictions under a latency budget | CatBoost, oblivious trees index rather than walk |
| GPU training, wide platform support, the largest ecosystem | XGBoost |
| You want the deployment story to be boring | XGBoost, the most widely supported at inference |

| Dial | XGBoost | LightGBM | CatBoost | HistGB |
|---|---|---|---|---|
| Number of trees | `n_estimators` | `n_estimators` | `iterations` | `max_iter` |
| Step size | `learning_rate` | `learning_rate` | `learning_rate` | `learning_rate` |
| Tree size | `max_depth` | `num_leaves` | `depth` | `max_leaf_nodes` |
| Bins | `max_bin` | `max_bin` | `border_count` | `max_bins` |
| Categorical | `enable_categorical=True` | pandas `category` dtype | `cat_features=[...]` | `categorical_features=[...]` |
| Quiet | `verbosity=0` | `verbose=-1` | `verbose=0` | quiet already |

## What to remember

1. All four are the same algorithm. The differences are engineering, and the
   engineering has mostly converged.
2. Histogram binning is the change that made boosting practical. It removes the
   per-node sort and caps the split search at `max_bin` positions.
3. Sibling subtraction gets a node's histogram for free from its parent and its
   sibling. Half the work at every level.
4. Level-wise grows balanced, leaf-wise grows lopsided and deep, oblivious
   grows one condition per level. Leaf-wise fits the training data hardest and
   overfits small data fastest.
5. Ordinal codes invent an order that is not in the data. One-hot is honest but
   makes the tree work harder and gets expensive as cardinality rises. Native
   handling splits on sets of categories in one step.
6. Target encoding leaks, because the row you are encoding is inside the mean.
   CatBoost's ordered statistics only averages over rows earlier in a random
   permutation, so the leak closes by construction.
7. Benchmarks at default settings measure the defaults. Force the same tree
   count, learning rate and leaf budget before you compare anything.
8. Tuning moves the accuracy more than the choice of library does. Choose on
   fitting speed, prediction speed, categorical support and how much you enjoy
   the API, then go and tune.

---

**Made by Elyes Lounissi** ·
[LinkedIn](https://www.linkedin.com/in/elyes-lounissi/) ·
[pilot.tun@gmail.com](mailto:pilot.tun@gmail.com)

Next: [04-07 Stacking and voting](../07-stacking-and-voting/) ·
Back to [the curriculum](../../CURRICULUM.md)

`#MachineLearning` `#XGBoost` `#LightGBM` `#CatBoost` `#GradientBoosting`
`#Python` `#ScikitLearn` `#DataScience` `#MLTutorial` `#Ensemble`